# Map External Lab Compound Names to CID Labels

Use this notebook as the first bridge between a separate lab compound-name file and the ChemSynonymizer synonym table. It performs exact matching after the same name normalization used by the retrieval pipeline, assigns candidate CID labels, and exports matched, ambiguous, and unmatched rows for downstream evaluation.

Expected reference CSV columns: `cid`, `name`.

Expected lab CSV columns for this first pass: `metabolite_name`, `refmet_name`.

In [1]:
# Colab setup. Run once per runtime.
# The pandas pin avoids the pandas 3.x incompatibility warnings in Colab.
%pip install -q "pandas==2.2.2" "numpy>=1.26.4,<2.1" "PyYAML>=6.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# No Drive mount is required for this notebook.
# It expects both CSV files to live in the repo's data/ folder.

In [3]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent, cwd / 'chem_synonymizer']
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer. Start Jupyter from the repo or notebooks folder.')


REPO_DIR = find_repo_dir()
DATA_DIR = REPO_DIR / 'data'
REFERENCE_CSV_FILENAME = 'CID-Synonym-full.csv'
# To use the smaller filtered reference instead, set REFERENCE_CSV_FILENAME = 'good.csv'.
LAB_CSV_PATH = DATA_DIR / 'allmetnames_062626.csv'
REFERENCE_CSV_PATH = DATA_DIR / REFERENCE_CSV_FILENAME

sys.path.insert(0, str(REPO_DIR / 'src'))
from chem_synonymizer.normalize import normalize_name

CID_COL = 'cid'
NAME_COL = 'name'
LAB_NAME_COLUMNS = ['metabolite_name', 'refmet_name']
PREFERRED_EXPORT_NAME_COLUMN = 'refmet_name'
REFERENCE_CHUNKSIZE = 2_000_000
REBUILD_TARGETED_LOOKUP = False

OUTPUT_DIR = REPO_DIR / 'results' / 'lab_cid_mapping'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGETED_LOOKUP_CACHE_PATH = OUTPUT_DIR / f'targeted_lookup_{LAB_CSV_PATH.stem}__{REFERENCE_CSV_PATH.stem}.csv'

print('Repo:', REPO_DIR)
print('Lab CSV:', LAB_CSV_PATH)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Output:', OUTPUT_DIR)

Repo: C:\Users\naran\Downloads\currentProj\chem_synonymizer
Lab CSV: C:\Users\naran\Downloads\currentProj\chem_synonymizer\data\allmetnames_062626.csv
Reference CSV: C:\Users\naran\Downloads\currentProj\chem_synonymizer\data\CID-Synonym-full.csv
Output: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping


## Preview Inputs

This preview reads only a few rows so you can confirm paths and column names before loading the large synonym table.

In [4]:
def require_path(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Path does not exist: {path}')


def preview_csv(path: Path, nrows: int = 8) -> pd.DataFrame:
    require_path(path)
    try:
        df = pd.read_csv(path, dtype='string', nrows=nrows, on_bad_lines='skip')
    except Exception:
        df = pd.read_csv(path, dtype='string', nrows=nrows, engine='python', on_bad_lines='skip')
    print(path)
    print('columns:', list(df.columns))
    display(df)
    return df


lab_preview = preview_csv(LAB_CSV_PATH)
reference_preview = preview_csv(REFERENCE_CSV_PATH)

C:\Users\naran\Downloads\currentProj\chem_synonymizer\data\allmetnames_062626.csv
columns: ['metabolite_name', 'refmet_name']


,metabolite_name,refmet_name
0,"""delphinidin 3-O-(6""""""""""""""""""""""""""""""""-O-malyl-be...",<NA>
1,"( )-2-(1-Methylpropyl)-4,6-dinitrophenol","2-(1-Methylpropyl)-4,6-dinitrophenol"
2,"(-)-11-Hydroxy-9,15,16-trioxooctadecanoic acid",<NA>
3,"(-)-2,3-Dihydro-6-methoxy-2-methyl-naphtho[1,2...",<NA>
4,"(-)-9,10-Dihydrojasmonic acid",<NA>
5,"(-)-Epigallocatechin 3-gallate 7-glucoside 4""""...",<NA>
6,"({3,4,5-trihydroxy-6-[5,6,7-trihydroxy-2-(4-hy...",<NA>
7,"({6-[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-4-o...",<NA>


C:\Users\naran\Downloads\currentProj\chem_synonymizer\data\CID-Synonym-full.csv
columns: ['cid', 'name']


,cid,name
0,1,Acetyl-DL-carnitine
1,1,14992-62-2
2,1,DL-O-Acetylcarnitine
3,1,07OP6H4V4A
4,1,DTXSID2048117
5,1,RefChem:1076018
6,1,DTXCID5028089
7,1,DL-Acetylcarnitine


## Build Targeted Reference Name-to-CID Lookup

The lab file is normalized first. Then the 114M-row reference synonym table is streamed in chunks and filtered to only the normalized names that appear in the lab file. This avoids building a full reference lookup for names that are not needed.

In [5]:
def normalize_series(values: pd.Series) -> pd.Series:
    return (
        values.fillna('')
        .astype('string')
        .str.normalize('NFKC')
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def load_lab_long(path: Path, name_columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    require_path(path)
    lab = pd.read_csv(path, dtype='string', engine='python', on_bad_lines='skip')
    available_name_columns = [col for col in name_columns if col in lab.columns]
    if not available_name_columns:
        raise ValueError(f'None of {name_columns} were found. Available columns: {list(lab.columns)}')

    lab = lab.reset_index(drop=True).copy()
    lab['_lab_row_id'] = np.arange(len(lab), dtype=np.int64)

    long = lab[['_lab_row_id'] + available_name_columns].melt(
        id_vars=['_lab_row_id'],
        value_vars=available_name_columns,
        var_name='source_column',
        value_name='lab_name',
    )
    long = long.dropna(subset=['lab_name']).copy()
    long['lab_name'] = long['lab_name'].astype(str).str.strip()
    long = long[long['lab_name'].str.len() > 0].copy()
    long['normalized_name'] = normalize_series(long['lab_name'])
    long = long[long['normalized_name'].str.len() > 0].copy()
    long = long.drop_duplicates(['_lab_row_id', 'source_column', 'normalized_name'])
    return lab, long, available_name_columns


def csv_chunks(path: Path, usecols: list[str], chunksize: int):
    try:
        yield from pd.read_csv(
            path,
            usecols=usecols,
            dtype='string',
            chunksize=chunksize,
            on_bad_lines='skip',
        )
    except Exception as exc:
        print(f'Default CSV parser failed ({exc}); retrying with the Python parser.')
        yield from pd.read_csv(
            path,
            usecols=usecols,
            dtype='string',
            chunksize=chunksize,
            engine='python',
            on_bad_lines='skip',
        )


def join_unique(values: pd.Series, sep: str = '|') -> str:
    seen = set()
    out = []
    for value in values.dropna():
        for item in str(value).split(sep):
            item = item.strip()
            if item and item not in seen:
                seen.add(item)
                out.append(item)
    return sep.join(out)


def count_joined(value: object, sep: str = '|') -> int:
    if pd.isna(value):
        return 0
    text = str(value).strip()
    if not text:
        return 0
    return len([part for part in text.split(sep) if part.strip()])


def build_targeted_reference_lookup(
    reference_csv_path: Path,
    target_normalized_names: set[str],
    cid_col: str = 'cid',
    name_col: str = 'name',
    chunksize: int = 1_000_000,
    cache_path: Path | None = None,
    rebuild_cache: bool = False,
) -> tuple[pd.DataFrame, dict]:
    require_path(reference_csv_path)
    if not target_normalized_names:
        raise ValueError('No lab names were available for targeted lookup.')

    if cache_path is not None and cache_path.exists() and not rebuild_cache:
        lookup = pd.read_csv(cache_path, dtype='string')
        lookup['n_candidate_cids'] = lookup['candidate_cids'].map(count_joined).astype('int32')
        summary = {
            'loaded_from_cache': True,
            'target_lab_normalized_names': int(len(target_normalized_names)),
            'reference_rows_read': None,
            'matched_reference_name_cid_pairs': int(lookup['n_candidate_cids'].sum()),
            'unique_matched_normalized_names': int(len(lookup)),
            'ambiguous_matched_normalized_names': int((lookup['n_candidate_cids'] > 1).sum()),
        }
        return lookup, summary

    pairs = []
    total_rows = 0
    kept_pairs = 0

    for i, chunk in enumerate(csv_chunks(reference_csv_path, [cid_col, name_col], chunksize), start=1):
        total_rows += len(chunk)
        chunk = chunk.rename(columns={cid_col: 'cid', name_col: 'name'})
        chunk = chunk.dropna(subset=['cid', 'name']).copy()
        chunk['normalized_name'] = normalize_series(chunk['name'])
        chunk = chunk[chunk['normalized_name'].str.len() > 0]
        chunk = chunk[chunk['normalized_name'].isin(target_normalized_names)]
        if chunk.empty:
            if i == 1 or i % 5 == 0:
                print(f'processed chunk {i:,}; source rows={total_rows:,}; matched pairs kept so far={kept_pairs:,}')
            continue
        chunk = chunk[['normalized_name', 'cid']].drop_duplicates()
        kept_pairs += len(chunk)
        pairs.append(chunk)

        if i == 1 or i % 5 == 0:
            print(f'processed chunk {i:,}; source rows={total_rows:,}; matched pairs kept so far={kept_pairs:,}')

    if not pairs:
        lookup = pd.DataFrame(columns=['normalized_name', 'candidate_cids', 'n_candidate_cids'])
        summary = {
            'loaded_from_cache': False,
            'target_lab_normalized_names': int(len(target_normalized_names)),
            'reference_rows_read': int(total_rows),
            'matched_reference_name_cid_pairs': 0,
            'unique_matched_normalized_names': 0,
            'ambiguous_matched_normalized_names': 0,
        }
        return lookup, summary

    ref_pairs = pd.concat(pairs, ignore_index=True).drop_duplicates()
    lookup = (
        ref_pairs.groupby('normalized_name', sort=False)['cid']
        .agg(join_unique)
        .reset_index(name='candidate_cids')
    )
    lookup['n_candidate_cids'] = lookup['candidate_cids'].map(count_joined).astype('int32')
    if cache_path is not None:
        lookup.to_csv(cache_path, index=False)

    summary = {
        'loaded_from_cache': False,
        'target_lab_normalized_names': int(len(target_normalized_names)),
        'reference_rows_read': int(total_rows),
        'matched_reference_name_cid_pairs': int(len(ref_pairs)),
        'unique_matched_normalized_names': int(len(lookup)),
        'ambiguous_matched_normalized_names': int((lookup['n_candidate_cids'] > 1).sum()),
    }
    return lookup, summary


lab_df, lab_long, available_lab_name_columns = load_lab_long(LAB_CSV_PATH, LAB_NAME_COLUMNS)
target_normalized_names = set(lab_long['normalized_name'].dropna().astype(str).unique())
print('lab rows:', f'{len(lab_df):,}')
print('lab name values:', f'{len(lab_long):,}')
print('unique normalized lab names:', f'{len(target_normalized_names):,}')
print('available lab name columns:', available_lab_name_columns)

reference_lookup, reference_summary = build_targeted_reference_lookup(
    REFERENCE_CSV_PATH,
    target_normalized_names=target_normalized_names,
    cid_col=CID_COL,
    name_col=NAME_COL,
    chunksize=REFERENCE_CHUNKSIZE,
    cache_path=TARGETED_LOOKUP_CACHE_PATH,
    rebuild_cache=REBUILD_TARGETED_LOOKUP,
)

display(pd.DataFrame([reference_summary]).T.rename(columns={0: 'value'}))
display(reference_lookup.head())

lab rows: 443,540
lab name values: 755,091
unique normalized lab names: 467,549
available lab name columns: ['metabolite_name', 'refmet_name']
processed chunk 1; source rows=2,000,000; matched pairs kept so far=16,918
processed chunk 5; source rows=10,000,000; matched pairs kept so far=35,988
processed chunk 10; source rows=20,000,000; matched pairs kept so far=46,945
processed chunk 15; source rows=30,000,000; matched pairs kept so far=63,337
processed chunk 20; source rows=40,000,000; matched pairs kept so far=68,933
processed chunk 25; source rows=50,000,000; matched pairs kept so far=80,124
processed chunk 30; source rows=60,000,000; matched pairs kept so far=82,128
processed chunk 35; source rows=70,000,000; matched pairs kept so far=84,554
processed chunk 40; source rows=80,000,000; matched pairs kept so far=86,467
processed chunk 45; source rows=90,000,000; matched pairs kept so far=89,596
processed chunk 50; source rows=100,000,000; matched pairs kept so far=106,277
processed c

,value
loaded_from_cache,False
target_lab_normalized_names,467549
reference_rows_read,114674081
matched_reference_name_cid_pairs,114775
unique_matched_normalized_names,109086
ambiguous_matched_normalized_names,3627


,normalized_name,candidate_cids,n_candidate_cids
0,acetyl-dl-carnitine,1,1
1,dl-acetylcarnitine,1,1
2,acetyl carnitine,1|7045767,2
3,"2,3-dihydroxy-2,3-dihydrobenzoic acid",3,1
4,1-aminopropan-2-ol,4,1


## Map Lab Names to Candidate CIDs

Each lab row is converted into a long format over the configured name columns. A row is marked `exact_unique_cid` only when all matched names collapse to one candidate CID. Rows with multiple candidate CIDs are preserved for review.

In [6]:
# lab_df and lab_long were loaded before the targeted reference scan.
lab_matches_long = lab_long.merge(reference_lookup, on='normalized_name', how='left')
lab_matches_long['is_exact_match'] = lab_matches_long['candidate_cids'].notna()

print('lab rows:', f'{len(lab_df):,}')
print('lab name values:', f'{len(lab_long):,}')
print('available lab name columns:', available_lab_name_columns)
display(lab_matches_long.head(20))

lab rows: 443,540
lab name values: 755,091
available lab name columns: ['metabolite_name', 'refmet_name']


,_lab_row_id,source_column,lab_name,normalized_name,candidate_cids,n_candidate_cids,is_exact_match
0,0,metabolite_name,"""delphinidin 3-O-(6""""""""""""""""""""""""""""""""-O-malyl-be...","""delphinidin 3-o-(6""""""""""""""""""""""""""""""""-o-malyl-be...",<NA>,NaN,False
1,1,metabolite_name,"( )-2-(1-Methylpropyl)-4,6-dinitrophenol","( )-2-(1-methylpropyl)-4,6-dinitrophenol",<NA>,NaN,False
2,2,metabolite_name,"(-)-11-Hydroxy-9,15,16-trioxooctadecanoic acid","(-)-11-hydroxy-9,15,16-trioxooctadecanoic acid",131752496,1.0,True
3,3,metabolite_name,"(-)-2,3-Dihydro-6-methoxy-2-methyl-naphtho[1,2...","(-)-2,3-dihydro-6-methoxy-2-methyl-naphtho[1,2...",<NA>,NaN,False
4,4,metabolite_name,"(-)-9,10-Dihydrojasmonic acid","(-)-9,10-dihydrojasmonic acid",644120,1.0,True
5,5,metabolite_name,"(-)-Epigallocatechin 3-gallate 7-glucoside 4""""...","(-)-epigallocatechin 3-gallate 7-glucoside 4""""...",<NA>,NaN,False
6,6,metabolite_name,"({3,4,5-trihydroxy-6-[5,6,7-trihydroxy-2-(4-hy...","({3,4,5-trihydroxy-6-[5,6,7-trihydroxy-2-(4-hy...",<NA>,NaN,False
7,7,metabolite_name,"({6-[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-4-o...","({6-[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-4-o...",<NA>,NaN,False
8,8,metabolite_name,"({6-[5,7-dihydroxy-2-(3-hydroxyphenyl)-4-oxo-3...","({6-[5,7-dihydroxy-2-(3-hydroxyphenyl)-4-oxo-3...",<NA>,NaN,False
9,9,metabolite_name,"({6-[5,7-dihydroxy-2-(3-hydroxyphenyl)-4-oxo-6...","({6-[5,7-dihydroxy-2-(3-hydroxyphenyl)-4-oxo-6...",<NA>,NaN,False


In [7]:
matched_only = lab_matches_long.copy()
unmatched_mask = ~matched_only['is_exact_match']
matched_only.loc[unmatched_mask, ['candidate_cids', 'source_column', 'lab_name', 'normalized_name']] = pd.NA

row_matches = (
    matched_only.groupby('_lab_row_id', dropna=False)
    .agg(
        candidate_cids=('candidate_cids', join_unique),
        matched_source_columns=('source_column', join_unique),
        matched_lab_names=('lab_name', join_unique),
        matched_normalized_names=('normalized_name', join_unique),
    )
    .reset_index()
)
row_matches['n_candidate_cids'] = row_matches['candidate_cids'].map(count_joined).astype('int32')
row_matches['mapping_status'] = np.select(
    [
        row_matches['n_candidate_cids'].eq(0),
        row_matches['n_candidate_cids'].eq(1),
        row_matches['n_candidate_cids'].gt(1),
    ],
    ['unmatched', 'exact_unique_cid', 'exact_ambiguous_cid'],
    default='unmatched',
)

mapped_lab = lab_df.merge(row_matches, on='_lab_row_id', how='left')
fill_text_cols = ['candidate_cids', 'matched_source_columns', 'matched_lab_names', 'matched_normalized_names']
mapped_lab[fill_text_cols] = mapped_lab[fill_text_cols].fillna('')
mapped_lab['n_candidate_cids'] = mapped_lab['n_candidate_cids'].fillna(0).astype('int32')
mapped_lab['mapping_status'] = mapped_lab['mapping_status'].fillna('unmatched')

mapping_summary = {
    'lab_rows': int(len(mapped_lab)),
    'lab_name_values_used': int(len(lab_long)),
    'matched_rows': int((mapped_lab['n_candidate_cids'] > 0).sum()),
    'unique_cid_rows': int(mapped_lab['mapping_status'].eq('exact_unique_cid').sum()),
    'ambiguous_cid_rows': int(mapped_lab['mapping_status'].eq('exact_ambiguous_cid').sum()),
    'unmatched_rows': int(mapped_lab['mapping_status'].eq('unmatched').sum()),
    'unique_assigned_cids': int(mapped_lab.loc[mapped_lab['mapping_status'].eq('exact_unique_cid'), 'candidate_cids'].nunique()),
}
mapping_summary['matched_row_pct'] = round(100 * mapping_summary['matched_rows'] / max(mapping_summary['lab_rows'], 1), 3)
mapping_summary['unique_cid_row_pct'] = round(100 * mapping_summary['unique_cid_rows'] / max(mapping_summary['lab_rows'], 1), 3)

display(pd.DataFrame([mapping_summary]).T.rename(columns={0: 'value'}))
display(mapped_lab['mapping_status'].value_counts(dropna=False).rename_axis('mapping_status').reset_index(name='rows'))
display(mapped_lab.head(20))

,value
lab_rows,443540.000
lab_name_values_used,755091.000
matched_rows,233002.000
unique_cid_rows,216792.000
ambiguous_cid_rows,16210.000
unmatched_rows,210538.000
unique_assigned_cids,81957.000
matched_row_pct,52.532
unique_cid_row_pct,48.878


,mapping_status,rows
0,exact_unique_cid,216792
1,unmatched,210538
2,exact_ambiguous_cid,16210


,metabolite_name,refmet_name,_lab_row_id,candidate_cids,matched_source_columns,matched_lab_names,matched_normalized_names,n_candidate_cids,mapping_status
0,"""delphinidin 3-O-(6""""""""""""""""""""""""""""""""-O-malyl-be...",<NA>,0,,,,,0,unmatched
1,"( )-2-(1-Methylpropyl)-4,6-dinitrophenol","2-(1-Methylpropyl)-4,6-dinitrophenol",1,6950,refmet_name,"2-(1-Methylpropyl)-4,6-dinitrophenol","2-(1-methylpropyl)-4,6-dinitrophenol",1,exact_unique_cid
2,"(-)-11-Hydroxy-9,15,16-trioxooctadecanoic acid",<NA>,2,131752496,metabolite_name,"(-)-11-Hydroxy-9,15,16-trioxooctadecanoic acid","(-)-11-hydroxy-9,15,16-trioxooctadecanoic acid",1,exact_unique_cid
3,"(-)-2,3-Dihydro-6-methoxy-2-methyl-naphtho[1,2...",<NA>,3,,,,,0,unmatched
4,"(-)-9,10-Dihydrojasmonic acid",<NA>,4,644120,metabolite_name,"(-)-9,10-Dihydrojasmonic acid","(-)-9,10-dihydrojasmonic acid",1,exact_unique_cid
5,"(-)-Epigallocatechin 3-gallate 7-glucoside 4""""...",<NA>,5,,,,,0,unmatched
6,"({3,4,5-trihydroxy-6-[5,6,7-trihydroxy-2-(4-hy...",<NA>,6,,,,,0,unmatched
7,"({6-[2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-4-o...",<NA>,7,,,,,0,unmatched
8,"({6-[5,7-dihydroxy-2-(3-hydroxyphenyl)-4-oxo-3...",<NA>,8,,,,,0,unmatched
9,"({6-[5,7-dihydroxy-2-(3-hydroxyphenyl)-4-oxo-6...",<NA>,9,,,,,0,unmatched


## Save CID Mapping Outputs

The `lab_labeled_for_evaluation.csv` export is the clean exact-match subset with one CID per row. Use the ambiguous and unmatched exports for manual review or a second-pass semantic retrieval experiment.

In [8]:
def choose_export_name(row: pd.Series, preferred_column: str, available_columns: list[str]) -> str:
    ordered = []
    if preferred_column in available_columns:
        ordered.append(preferred_column)
    ordered.extend([col for col in available_columns if col not in ordered])
    for col in ordered:
        value = row.get(col, pd.NA)
        if not pd.isna(value) and str(value).strip():
            return str(value).strip()
    return ''


exact_unique = mapped_lab[mapped_lab['mapping_status'].eq('exact_unique_cid')].copy()
if exact_unique.empty:
    labeled_for_eval = pd.DataFrame(columns=['cid', 'name', 'source_row_id'])
else:
    labeled_for_eval = pd.DataFrame({
        'cid': exact_unique['candidate_cids'].astype(str),
        'name': exact_unique.apply(
            choose_export_name,
            axis=1,
            preferred_column=PREFERRED_EXPORT_NAME_COLUMN,
            available_columns=available_lab_name_columns,
        ),
        'source_row_id': exact_unique['_lab_row_id'].astype('int64'),
    })

output_paths = {
    'mapped_lab': OUTPUT_DIR / 'lab_names_with_candidate_cids.csv',
    'long_matches': OUTPUT_DIR / 'lab_name_column_matches.csv',
    'labeled_for_eval': OUTPUT_DIR / 'lab_labeled_for_evaluation.csv',
    'ambiguous': OUTPUT_DIR / 'lab_names_ambiguous_cids.csv',
    'unmatched': OUTPUT_DIR / 'lab_names_unmatched.csv',
    'summary': OUTPUT_DIR / 'mapping_summary.json',
}

mapped_lab.to_csv(output_paths['mapped_lab'], index=False)
lab_matches_long.to_csv(output_paths['long_matches'], index=False)
labeled_for_eval.to_csv(output_paths['labeled_for_eval'], index=False)
mapped_lab[mapped_lab['mapping_status'].eq('exact_ambiguous_cid')].to_csv(output_paths['ambiguous'], index=False)
mapped_lab[mapped_lab['mapping_status'].eq('unmatched')].to_csv(output_paths['unmatched'], index=False)

with output_paths['summary'].open('w', encoding='utf-8') as f:
    json.dump(
        {
            'reference_summary': reference_summary,
            'mapping_summary': mapping_summary,
            'lab_csv_path': str(LAB_CSV_PATH),
            'reference_csv_path': str(REFERENCE_CSV_PATH),
            'lab_name_columns': available_lab_name_columns,
        },
        f,
        indent=2,
    )

for label, path in output_paths.items():
    print(f'{label}: {path}')

display(labeled_for_eval.head(20))

mapped_lab: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping\lab_names_with_candidate_cids.csv
long_matches: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping\lab_name_column_matches.csv
labeled_for_eval: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping\lab_labeled_for_evaluation.csv
ambiguous: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping\lab_names_ambiguous_cids.csv
unmatched: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping\lab_names_unmatched.csv
summary: C:\Users\naran\Downloads\currentProj\chem_synonymizer\results\lab_cid_mapping\mapping_summary.json


,cid,name,source_row_id
1,6950,"2-(1-Methylpropyl)-4,6-dinitrophenol",1
2,131752496,"(-)-11-Hydroxy-9,15,16-trioxooctadecanoic acid",2
4,644120,"(-)-9,10-Dihydrojasmonic acid",4
11,6442246,"(+)-(R)-2-(2,4-Difluorophenyl)-1-(3-((E)-4-(2,...",11
14,131752469,"(1(10)E,4a,5E)-1(10),5-Germacradiene-12-acetox...",14
15,131751160,"(1(10)E,4E,6a,9b)-9-(2-Methylpropanoyloxy)-1(1...",15
16,581958,"(1,2,3-Trimethyl-2-cyclopropen-1-yl)benzene",16
18,9548877,9S-HpODE,18
19,44559173,"(10E,15Z)-9,12,13-Trihydroxy-10,15-octadecadie...",19
22,156961007,"(10Z,12Z)-Octadeca-10,12-Dienoylcarnitine",22


## Next Experiment Direction

Use `lab_labeled_for_evaluation.csv` as the external exact-match evaluation subset. The unmatched rows are the best candidates for a second pass with the semantic retriever: embed the lab names as queries, retrieve nearest known synonyms, and compare high-confidence predictions against manual review or future CID labels.